# Germanene bubble: radius sweep at height 1 nm

Compare radii **2, 3, 4 and 5 nm** at a fixed peak height **1 nm**, using the same TBPLaS model as the height sweep:

$$z(X,Y)=h\exp\left[-\frac{X^2+Y^2}{2(R/3)^2}\right],\qquad h=1\,\mathrm{nm}.$$

Radius means **three Gaussian standard deviations**. Changing radius at fixed height changes both width and slope. Material parameters, global pz axes and fixed SOC are inherited from `model.py`; the prescribed geometry is not mechanically relaxed.

**The periodic simulation cell is fixed across all radii.** Its nominal side is six times the largest radius (30 nm for the defaults). Since `ExportParams` normally sizes the cell from `lfac * radius`, this notebook adjusts `lfac` inversely with radius to keep the atom count, boundary conditions, physical site ordering and spectral sampling unchanged. Increase `CELL_LFAC` for a larger common cell. The 3 nm case therefore uses a larger cell than the default height-sweep notebook.

Line spectra use a common physical line extending to ±1.5 times the largest radius (±7.5 nm by default), with target spacing no greater than 0.375 nm. The bubble ROI contains both A and B atoms within each bubble's own radius, so its atom count varies. Each ROI has a flat-sheet reference on exactly the same atoms and phases. Densities are **averages per spin orbital**, not integrated spectral weight; different ROI areas must be considered when comparing their averages.

Full settings: 1201 energies from −1 to +1 eV, 10 meV broadening, 1500 recursion steps and eight random vectors. `HB_QUICK=1` reduces spectral sampling for a smoke check and writes to a separate folder. This parameter sweep does not establish finite-size, recursion or physical-model convergence.

Intrinsic Rashba is explicitly disabled in this sweep: $\lambda_{R2}=0$, while $\lambda_{\mathrm{SO}}=100\,\mathrm{meV}$ remains unchanged. All saved filenames begin with `R_`. Figures use 300 DPI and lettered panels.

In [ ]:
# Resolve the shared model and output root from this notebook or the project root.
from pathlib import Path
import os
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'model.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import os
import json
import csv
from dataclasses import asdict, replace
from pathlib import Path
from time import perf_counter
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from model import (ExportParams, build_model, cell_display, pseudofield,
                   gaussian_height, line_sites, line_ldos, bubble_sites,
                   roi_ldos, total_dos)

fig_size0 = (6.8, 4.2)

params = {
    "backend": "ps",
    "axes.labelsize": 13,
    "font.size": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "text.usetex": True,
    "figure.figsize": fig_size0,
    "font.family": "serif",
    "font.serif": "Computer Modern Roman",
    "legend.frameon": True,
    "savefig.dpi": 300,
}

plt.rcParams.update(params)


def label_panels(axes):
    """Label data panels left-to-right, then top-to-bottom; omit colorbars."""
    panels = np.asarray(axes, dtype=object).ravel()
    if len(panels) > 1:
        for index, ax in enumerate(panels):
            ax.set_title(f'({chr(ord("a") + index)})', loc='left',
                         fontsize=13, fontweight='bold')

RADII_NM = np.array([2.0, 3.0, 4.0, 5.0])
HEIGHT_NM = 1.0
CELL_LFAC = 3.0  # common nominal cell side = 2 * CELL_LFAC * max(RADII_NM)
LINE_STEP_NM = 0.375
QUICK = os.getenv('HB_QUICK', '0') == '1'
SEED = 20260923
assert RADII_NM.ndim == 1 and len(RADII_NM) > 0
assert np.all(np.isfinite(RADII_NM)) and np.all(RADII_NM > 0)
assert len(np.unique(RADII_NM)) == len(RADII_NM)
assert np.isfinite(HEIGHT_NM) and HEIGHT_NM >= 0
assert CELL_LFAC >= 2 and LINE_STEP_NM > 0
max_radius = float(RADII_NM.max())
base = ExportParams(radius=max_radius, height=HEIGHT_NM, lfac=CELL_LFAC, lambda_r2_mev=0.0)

def params_for_radius(radius_nm):
    return replace(base, radius=float(radius_nm),
                   lfac=CELL_LFAC * max_radius / float(radius_nm))

assert all(params_for_radius(r).dimensions == base.dimensions for r in RADII_NM)
energy = base.energy
depth = 250 if QUICK else base.recursion_depth
line_half_length = 1.5 * max_radius
points = 9 if QUICK else 2 * int(np.ceil(line_half_length / LINE_STEP_NM)) + 1
draws = 2 if QUICK else base.roi_random_vectors
out = Path('results/gaussian/06_radius_sweep') / ('quick' if QUICK else 'full')
out.mkdir(parents=True, exist_ok=True)
metadata = dict(base_parameters=asdict(base), radii_nm=RADII_NM.tolist(),
                height_nm=HEIGHT_NM, nominal_cell_side_nm=2*CELL_LFAC*max_radius,
                dimensions=list(base.dimensions), recursion_depth=depth,
                line_points=points, line_half_length_nm=line_half_length,
                random_vectors=draws, seed=SEED, quick=QUICK,
                radius_convention='R = 3 sigma', roi_sublattices='both', normalization='density per spin orbital',
                parameters_by_radius=[asdict(params_for_radius(r)) for r in RADII_NM])
(out / 'R_parameters.json').write_text(json.dumps(metadata, indent=2) + '\n')
print('QUICK CHECK' if QUICK else 'FULL SPECTRAL SETTINGS')
print(f'Radii = {RADII_NM} nm; fixed height = {HEIGHT_NM:g} nm')
print(f'Fixed cell dimensions: {base.dimensions}; {depth} recursion steps; '
      f'{points} line positions; {draws} random vectors')

## Common cell, flat reference and sampling

The flat Hamiltonian, line LDOS and full-cell DOS are computed once. Random phases are assigned to physical A- and B-site indices across the common sheet, then restricted and normalized separately within each radius-dependent ROI. Shared atoms retain the same phases for every radius. The flat ROI spectrum is recomputed for each mask; its paired stochastic samples are saved with that radius.

In [ ]:
t0 = perf_counter()
flat = build_model(base, flat=True)
ham_flat = flat.hamiltonian()
position, sites, targets = line_sites(flat, length=2*line_half_length, points=points,
                                     angle_deg=0, sublattice='both')
assert len(position) == points, 'Requested line spacing selected duplicate A/B pairs'
physical_sites = np.arange(len(flat.is_a))
phase_table = np.exp(2j * np.pi * np.random.default_rng(SEED).random((len(physical_sites), draws)))
flat_line = line_ldos(ham_flat, sites, energy, base.eta, depth)
flat_dos, flat_dos_error, flat_dos_draws = total_dos(
    ham_flat, energy, base.eta, depth, draws=draws, seed=SEED)
for values in (flat_line, flat_dos, flat_dos_error, flat_dos_draws):
    assert np.all(np.isfinite(values))
assert flat_line.min() >= -1e-12 and flat_dos.min() >= -1e-12
np.savez_compressed(out / 'R_flat_reference.npz', energy=energy, position=position,
                    sites=sites, targets=targets, physical_sites=physical_sites, is_a=flat.is_a, phases=phase_table,
                    display_xy=flat.display_xy, line_ldos=flat_line, dos=flat_dos,
                    dos_error=flat_dos_error, dos_draws=flat_dos_draws)
print(f'{flat.dim[0]} × {flat.dim[1]} cells; {ham_flat.shape[0]:,} spin orbitals; '
      f'flat line/DOS reference: {perf_counter()-t0:.1f} s')

## Calculate each radius

Each radius is saved when it finishes. Checks enforce the same cell, centre, physical coordinates and line-site indices across the sweep. The ROI changes with radius, and the flat ROI spectrum uses that same selection. Both PMF conventions from notebook 01 are retained; the main maps display Eq. (A11) as printed.

In [ ]:
results = []
for radius_nm in RADII_NM:
    start = perf_counter()
    p = params_for_radius(radius_nm)
    bubble = build_model(p)
    assert bubble.dim == flat.dim
    np.testing.assert_allclose(bubble.center, flat.center)
    np.testing.assert_allclose(bubble.display_xy, flat.display_xy)
    np.testing.assert_array_equal(bubble.is_a, flat.is_a)
    ham = bubble.hamiltonian()
    assert ham.shape == ham_flat.shape
    residual = ham - ham.getH()
    assert residual.nnz == 0 or np.max(np.abs(residual.data)) < 1e-12
    pos, selected, _ = line_sites(bubble, length=2*line_half_length, points=points,
                                  angle_deg=0, sublattice='both')
    np.testing.assert_array_equal(selected, sites)
    np.testing.assert_allclose(pos, position)
    roi = bubble_sites(bubble)
    assert np.any(bubble.is_a[roi]) and np.any(~bubble.is_a[roi])
    phase_rows = np.searchsorted(physical_sites, roi)
    np.testing.assert_array_equal(physical_sites[phase_rows], roi)
    phases = phase_table[phase_rows]
    flat_roi, flat_roi_error, flat_roi_draws = roi_ldos(
        ham_flat, roi, energy, p.eta, depth, phases)
    gauge = pseudofield(bubble)
    rho_line = line_ldos(ham, sites, energy, p.eta, depth)
    rho_roi, roi_error, roi_draws = roi_ldos(ham, roi, energy, p.eta, depth, phases)
    dos, dos_error, dos_draws = total_dos(ham, energy, p.eta, depth,
                                        draws=draws, seed=SEED)
    paired = roi_draws - flat_roi_draws
    result = dict(radius_nm=float(radius_nm), height_nm=HEIGHT_NM, roi=roi,
                  roi_count=len(roi), roi_a_count=int(np.count_nonzero(bubble.is_a[roi])),
                  roi_b_count=int(np.count_nonzero(~bubble.is_a[roi])),
                  roi_is_a=bubble.is_a[roi], phases=phases, x=gauge['x'], y=gauge['y'],
                  height=cell_display(bubble, bubble.height),
                  strain=cell_display(bubble, bubble.strain),
                  Bpaper=gauge['Bpaper'], Bconsistent=gauge['B'],
                  line_ldos=rho_line, roi_ldos=rho_roi, roi_error=roi_error,
                  roi_draws=roi_draws, flat_roi=flat_roi, flat_roi_error=flat_roi_error,
                  flat_roi_draws=flat_roi_draws, roi_difference=paired.mean(axis=1),
                  roi_difference_error=paired.std(axis=1, ddof=1)/np.sqrt(draws),
                  dos=dos, dos_error=dos_error, dos_draws=dos_draws)
    for key, value in result.items():
        assert np.all(np.isfinite(value)), f'Non-finite {key} at R={radius_nm}'
    for key in ('line_ldos', 'roi_ldos', 'flat_roi', 'dos'):
        assert np.min(result[key]) >= -1e-12, f'Negative density: {key}'
    tag = f'R{radius_nm:g}nm'.replace('.', 'p')
    np.savez_compressed(out / f'R_{tag}.npz', **result, energy=energy,
                        position=position, sites=sites,
                        parameters_json=json.dumps(asdict(p)), roi_sublattices='both', quick=QUICK,
                        recursion_depth=depth, random_vectors=draws)
    results.append(result)
    print(f'R = {radius_nm:g} nm; {len(roi)} ROI atoms; '
          f'finished in {perf_counter()-start:.1f} s', flush=True)
    del ham, bubble
print(f'All calculations finished in {(perf_counter()-t0)/60:.1f} min')
del ham_flat

## Geometry and PMF on common colour scales

Maps average over the four physical atoms per rectangular cell. Each row uses one colour scale across all radii, without percentile clipping, and all panels show the same physical window. Orange circles mark the different bubble radii.

In [ ]:
def map_extent(result):
    x, y = result['x'], result['y']
    dx, dy = np.diff(x).mean(), np.diff(y).mean()
    return [x[0]-dx/2, x[-1]+dx/2, y[0]-dy/2, y[-1]+dy/2]

fig, axes = plt.subplots(3, len(results), figsize=(3.75*len(results), 10),
                         squeeze=False, constrained_layout=True)
for row, (key, label, cmap) in enumerate([
        ('height', 'height (nm)', 'viridis'),
        ('strain', 'bond strain', 'magma'),
        ('Bpaper', 'PMF (T)', 'RdBu_r')]):
    vmin = min(np.min(r[key]) for r in results)
    vmax = max(np.max(r[key]) for r in results)
    if key == 'Bpaper':
        vmax = max(abs(vmin), abs(vmax))
        vmin = -vmax
    for col, r in enumerate(results):
        ax = axes[row, col]
        im = ax.imshow(r[key], origin='lower', extent=map_extent(r),
                       cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.add_patch(plt.Circle((0, 0), r['radius_nm'], fill=False, color='darkorange'))
        ax.set(xlim=(-line_half_length, line_half_length), ylim=(-line_half_length, line_half_length), aspect='equal',
               xlabel='x (nm)', ylabel='y (nm)', title=f"R = {r['radius_nm']:g} nm")
    fig.colorbar(im, ax=axes[row, :].tolist(), label=label, shrink=0.8)
label_panels(axes)
fig.savefig(out / 'R_geometry_pmf.png', dpi=300)
plt.show()

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(results)))
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
x_profile = np.linspace(-line_half_length, line_half_length, 601)
for r, color in zip(results, colors):
    ax.plot(x_profile, gaussian_height(x_profile, 0, params_for_radius(r['radius_nm'])),
            color=color, label=f"R = {r['radius_nm']:g} nm")
ax.set(xlabel='x (nm)', ylabel='height (nm)', title='Gaussian profiles along y = 0')
ax.legend()
fig.savefig(out / 'R_height_profiles.png', dpi=300)
plt.show()

## Line spectroscopy at fixed physical positions

Each spectrum averages the nearest A and B sites and their two spin projections. All maps share the same log colour scale and physical X positions. Dotted lines mark each bubble's ±R. The line endpoints are fixed outside the largest bubble; the centre, far endpoint and band-edge spatial traces are compared against the common flat reference.

The spatial trace is evaluated at $E=+0.1\,\mathrm{eV}$ (the positive flat-sheet band edge for the chosen intrinsic SOC), using the nearest saved energy-grid point for both bubble and flat curves.

In [ ]:
logs = [np.log10(np.maximum(r['line_ldos'], 1e-12)) for r in results]
vmin, vmax = min(z.min() for z in logs), max(z.max() for z in logs)
fig, axes = plt.subplots(1, len(results), figsize=(4*len(results), 4.5),
                         sharey=True, squeeze=False, constrained_layout=True)
axes = axes.ravel()
for ax, r, z in zip(axes, results, logs):
    im = ax.pcolormesh(position, energy, z, shading='nearest',
                       cmap='magma', vmin=vmin, vmax=vmax, rasterized=True)
    for edge in (-r['radius_nm'], r['radius_nm']):
        ax.axvline(edge, color='white', ls=':', lw=0.7, alpha=0.8)
    ax.set(xlabel='x (nm)', title=f"R = {r['radius_nm']:g} nm")
axes[0].set_ylabel('energy (eV)')
fig.colorbar(im, ax=axes.tolist(), label='log10(LDOS / (1/eV))')
label_panels(axes)
fig.savefig(out / 'R_line_ldos_maps.png', dpi=300)
plt.show()

centre = int(np.argmin(abs(position)))
zero = int(np.argmin(abs(energy)))  # retained for the E=0 CSV summary
TRACE_ENERGY_EV = 0.1
trace_index = int(np.argmin(abs(energy - TRACE_ENERGY_EV)))
trace_energy = float(energy[trace_index])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for r, color in zip(results, colors):
    label = f"R = {r['radius_nm']:g} nm"
    axes[0].plot(energy, r['line_ldos'][:, centre], color=color, label=label)
    axes[1].plot(energy, r['line_ldos'][:, 0], color=color, label=label)
    axes[2].plot(position, r['line_ldos'][trace_index], '.-', color=color, label=label)
for ax, index, title in zip(axes[:2], [centre, 0], ['bubble centre', 'far endpoint']):
    ax.plot(energy, flat_line[:, index], 'k--', lw=1, label='flat reference')
    ax.set(xlabel='energy (eV)', ylabel='LDOS (1/eV)', title=title)
axes[2].plot(position, flat_line[trace_index], 'k--', lw=1, label='flat reference')
axes[2].set(xlabel='x (nm)', ylabel=f'LDOS at E = {trace_energy:g} eV (1/eV)', title='band-edge trace')
axes[0].legend(fontsize=8)
label_panels(axes)
fig.savefig(out / 'R_local_spectra.png', dpi=300)
plt.show()

## Radius-dependent bubble regions and full-cell DOS

Each ROI is a disk of radius R including both A and B sublattices. Its mean LDOS is compared with a flat reference using the same atom indices and phases. Shading shows one standard error of the random-vector estimate. Bubble-minus-flat errors use paired differences; these do not quantify recursion or finite-size error. The full-cell DOS is normalized over the same number of spin orbitals for every radius.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4*len(results), 4.5),
                         sharey=True, squeeze=False, constrained_layout=True)
for ax, r, color in zip(axes.ravel(), results, colors):
    ax.plot(energy, r['roi_ldos'], color=color, label='bubble')
    ax.fill_between(energy, r['roi_ldos']-r['roi_error'],
                    r['roi_ldos']+r['roi_error'], color=color, alpha=0.15)
    ax.plot(energy, r['flat_roi'], 'k--', lw=1, label='flat')
    ax.fill_between(energy, r['flat_roi']-r['flat_roi_error'],
                    r['flat_roi']+r['flat_roi_error'], color='0.5', alpha=0.1)
    ax.set(xlabel='energy (eV)',
           title=f"R = {r['radius_nm']:g} nm")
axes[0,0].set_ylabel('mean DOS (1/eV)')
axes[0,0].legend(fontsize=8)
label_panels(axes)
fig.savefig(out / 'R_roi_spectra.png', dpi=300)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for r, color in zip(results, colors):
    for ax, key, error in zip(axes, ['roi_difference', 'dos'],
                              ['roi_difference_error', 'dos_error']):
        ax.plot(energy, r[key], color=color, label=f"R = {r['radius_nm']:g} nm")
        ax.fill_between(energy, r[key]-r[error], r[key]+r[error], color=color, alpha=0.12)
axes[0].axhline(0, color='0.5', lw=0.7)
axes[1].plot(energy, flat_dos, 'k--', lw=1, label='flat reference')
axes[0].set(xlabel='energy (eV)', ylabel='mean LDOS difference (1/eV)',
            title='bubble minus flat')
axes[1].set(xlabel='energy (eV)', ylabel='DOS per spin orbital (1/eV)',
            title='full-cell DOS: fixed cell')
for ax in axes:
    ax.legend(fontsize=8)
label_panels(axes)
fig.savefig(out / 'R_roi_difference_dos.png', dpi=300)
plt.show()

## Saved numerical summary

PMF extrema refer to cell-averaged maps. The CSV includes radius, fixed height, ROI atom count and representative observables. NPZ files retain both PMF conventions, stochastic samples, paired flat ROI references and simulation parameters; the common flat file retains line-site coordinates and the global A+B-site phase table.

In [ ]:
summary = []
for r in results:
    summary.append(dict(radius_nm=r['radius_nm'], height_nm=HEIGHT_NM,
                        roi_atoms=r['roi_count'], roi_a_atoms=r['roi_a_count'],
                        roi_b_atoms=r['roi_b_count'],
                        max_cell_strain=float(np.max(r['strain'])),
                        max_abs_Bpaper_T=float(np.max(np.abs(r['Bpaper']))),
                        max_abs_Bconsistent_T=float(np.max(np.abs(r['Bconsistent']))),
                        centre_ldos_E0=float(r['line_ldos'][zero, centre]),
                        far_ldos_E0=float(r['line_ldos'][zero, 0]),
                        roi_ldos_E0=float(r['roi_ldos'][zero])))
with (out / 'R_summary.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
print('radius (nm) | max cell strain | max |Bpaper| (T) | centre LDOS(E=0) (1/eV)')
for row in summary:
    print(f"{row['radius_nm']:11.1f} | {row['max_cell_strain']:15.5f} | "
          f"{row['max_abs_Bpaper_T']:16.3f} | {row['centre_ldos_E0']:.8f}")
print(f'Figures, arrays, parameters and CSV saved in {out.resolve()}')